<a href="https://colab.research.google.com/github/Gabriele-Raffaele/concept_gridlock/blob/master/DCG_convert_chunk1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Raw Readers

In [ ]:
!git clone https://github.com/Gabriele-Raffaele/openpilot.git

In [ ]:
%cd ..


In [ ]:
!pip install pycapnp
!pip install smbus2
!pip install lru-dict

In [ ]:
from __future__ import print_function
import os
import numpy as np
import sys
from tqdm import tqdm
import cv2
import platform
platform.architecture()
sys.path.append("/content/openpilot")
from tools.lib.logreader import LogReader
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tools.lib.framereader import FrameReader
import h5py

In [ ]:
!git clone "https://github.com/commaai/comma2k19.git"

In [ ]:
example_segment = '/content/comma2k19/Example_1/b0c9d2329ad1606b|2018-08-02--08-34-47/40/'
lr = LogReader(example_segment + 'raw_log.bz2')
# make list of logs
logs = list(lr)
[l.which() for l in logs[:50]]

In [ ]:
# We can extract frames from the video with framereader
# from openpilot tools, we look at frame 600
frame_index = 600

fr = FrameReader(example_segment + 'video.hevc')
#figsize(12,12)
#imshow(fr.get(frame_index, pix_fmt='rgb24')[0]);
#title('Frame 600 extracted from video with FrameReader', fontsize=25);

In [ ]:
def get_sample(p):
    frame_reader = FrameReader(p+'/video.hevc')
    logs = list(LogReader(p + '/raw_log.bz2'))

    angle = np.array([l.carState.steeringAngleDeg for l in logs if l.which() == 'carState'])[1::5][1::5]
    time = np.array([l.logMonoTime for l in logs if l.which() == 'carState'])[1::5][1::5]
    vEgo = np.array([l.carState.vEgo for l in logs if l.which() == 'carState'])[1::5][1::5]
    gas = np.array([l.carState.gas for l in logs if l.which() == 'carState'])[1::5][1::5]
    brake = np.array([l.carState.brake for l in logs if l.which() == 'carState'])[1::5][1::5]
    gps_times = np.load(p + '/global_pose/frame_gps_times')
    times = np.load(p + '/global_pose/frame_times')
    gas = np.array([l.carState.gas for l in logs if l.which() == 'carState'])[1::5][1::5]
    gaspressed = np.array([l.carState.gasPressed for l in logs if l.which() == 'carState'])[1::5][1::5]
    brake = np.array([l.carState.brake for l in logs if l.which() == 'carState'])[1::5][1::5]
    brake_pressed = np.array([l.carState.brakePressed for l in logs if l.which() == 'carState'])[1::5][1::5]

    enabled = np.array([l.carState.cruiseState.enabled for l in logs if l.which() == 'carState'])[1::5][1::5]
    speed = np.array([l.carState.cruiseState.speed for l in logs if l.which() == 'carState'])[1::5][1::5]
    speedOffset = np.array([l.carState.cruiseState.speedOffset for l in logs if l.which() == 'carState'])[1::5][1::5]
    standstill = np.array([l.carState.cruiseState.standstill for l in logs if l.which() == 'carState'])[1::5][1::5]
    nonAdaptive = np.array([l.carState.cruiseState.nonAdaptive for l in logs if l.which() == 'carState'])[1::5][1::5]
    speedCluster = np.array([l.carState.cruiseState.speedCluster for l in logs if l.which() == 'carState'])[1::5][1::5]

    leftBlinker = np.array([l.carState.leftBlinker for l in logs if l.which() == 'carState'])[1::5][1::5]
    rightBlinker = np.array([l.carState.rightBlinker for l in logs if l.which() == 'carState'])[1::5][1::5]
    #print(frame_reader.frame_count, gps_times.shape, times.shape)
    #print([l.carState for l in logs if l.which() == "carState"][0])
    #print([l.radarState for l in logs if l.which() == "radarState"][0])

    dist = np.array([l.radarState.leadOne.dRel for l in logs if l.which() == "radarState"])[1::5]
    if ((vEgo == 0).mean() > 0.2) or ((dist == 0).mean() > 0.2) or len(dist) <=230:
        return None
    images = []
    l = list(range(frame_reader.frame_count))
    if len(l) > 245:
        l = l[1::5]
    for idx in list(range(frame_reader.frame_count))[1::5]:
        image = np.array(frame_reader.get(idx, pix_fmt='rgb24')[0], dtype=np.float64)
        image = cv2.resize(image, dsize=(224, 224), interpolation=cv2.INTER_CUBIC)
        images.append(image)
    steady_state = ~gaspressed & ~brake_pressed & ~leftBlinker & ~rightBlinker
    last_idx = 0
    desired_gap = np.zeros(steady_state.shape)

    for i in range(len(steady_state)-1):
        if steady_state[i] == True:
            desired_gap[last_idx:i] = int(dist[i])
            last_idx = i

    sample = {
        'image': images,
        "CruiseStateenabled": enabled,
        "CruiseStatespeed": speed,
        "CruiseStatespeedOffset": speedOffset,
        "CruiseStatestandstill": standstill,
        "CruiseStatenonAdaptive": nonAdaptive,
        "CruiseStatespeedCluster": speedCluster,
        'leftBlinker': leftBlinker,
        'rightBlinker': rightBlinker,
        "gas": gas,
        "gaspressed": gaspressed,
        "brake": brake,
        "brakepressed": brake_pressed,
        'angle': angle,
        'time': time,
        'gas': gas,
        'vEgo': vEgo,
        'brake': brake,
        'dist': dist,
        'desired_dist': desired_gap,
        }
    return sample if not ((desired_gap == 0).mean() > 0.2) else None

In [ ]:
def save_h5py(i, sample, h):
    group = h.create_group(str(i))
    for col in sample.keys():
            dt = np.float32 if col != 'image' else int#
            dataset_name = col #groups are divided by '/'
            a = list(sample[col])
            group.create_dataset(dataset_name, data = np.asarray(a, dtype=dt),
                    #compression_opts=9,
                    #chunks=(164, 20, 20, 3),
                    compression='lzf')

In [ ]:

# Vai nella cartella
%cd /content/dataset

# Scarica il primo chunk (~9GB)
!wget https://huggingface.co/datasets/commaai/comma2k19/resolve/main/Chunk_1.zip?download=true


In [ ]:
!mkdir -p /content/dataset
!mv "Chunk_1.zip?download=true" Chunk_1.zip
!mv Chunk_1.zip /content/dataset

In [ ]:
!wget https://huggingface.co/datasets/commaai/comma2k19/resolve/main/Chunk_2.zip?download=true

In [ ]:
!wget https://huggingface.co/datasets/commaai/comma2k19/resolve/main/Chunk_3.zip?download=true

In [ ]:
!unzip /content/dataset/Chunk_1.zip -d /content/dataset

In [ ]:
dataset_type = "train"
main_dir='/content/dataset/Chunk_1/'
hdf5_filename = "gas_and_brake_train_comma_chunk_1_w_imgs.hdf5"
h_train = h5py.File(hdf5_filename, 'w')
hdf5_filename = "gas_and_brake_val_comma_chunk_1_w_imgs.hdf5"
h_val = h5py.File(hdf5_filename, 'w')
hdf5_filename = "gas_and_brake_test_comma_chunk_1_w_imgs.hdf5"
h_test = h5py.File(hdf5_filename, 'w')

In [ ]:
for j, drive_sequence_path in tqdm(enumerate(os.listdir(main_dir))):
    if '.DS_Store ' in drive_sequence_path or not os.path.isdir(main_dir+"/"+drive_sequence_path): continue
    min_sequence_paths = os.listdir(main_dir+"/"+drive_sequence_path)
    if len(min_sequence_paths) < 3: continue
    min_sequence_path_test = main_dir+"/"+drive_sequence_path+"/"+min_sequence_paths[-2]
    min_sequence_paths_val = main_dir+"/"+drive_sequence_path+"/"+min_sequence_paths[-1]
    sample = get_sample(min_sequence_path_test)
    if sample != None:
        save_h5py(f"{drive_sequence_path}", sample, h_test)
    sample = get_sample(min_sequence_paths_val)

    if sample != None:
        save_h5py(f"{drive_sequence_path}", sample, h_val)
    for i, min_sequence in enumerate(min_sequence_paths[:-2]):
        if '.DS_Store ' in min_sequence or not os.path.isdir(main_dir+"/"+drive_sequence_path+'/'+min_sequence): continue
        p = main_dir+"/"+drive_sequence_path+"/"+min_sequence
        sample = get_sample(p)
        if sample != None:
            save_h5py(f"{drive_sequence_path}_{min_sequence}", sample, h_train)


In [ ]:
h_test.close()
h_train.close()
h_val.close()

In [ ]:
from google.colab import files
files.download("/content/gas_and_brake_train_comma_chunk_1_w_imgs.hdf5")
files.download("/content/gas_and_brake_val_comma_chunk_1_w_imgs.hdf5")
files.download("/content/gas_and_brake_test_comma_chunk_1_w_imgs.hdf5")

In [ ]:
%rm -r frames/

In [ ]:
import os
import cv2
from tqdm import tqdm
#from openpilot.tools.lib.video import FrameReader

# === CONFIGURA QUI ===
chunk_path = "/content/dataset/Chunk_1"  # Percorso del Chunk (estratto da ZIP)
output_base = "/content/frames"         # Dove salvare le immagini
fps_target = 12
output_size = (1280, 960)

os.makedirs(output_base, exist_ok=True)

# === GIRA SU TUTTE LE SEQUENZE ===
for drive in tqdm(os.listdir(chunk_path), desc="Processing drives"):
    drive_path = os.path.join(chunk_path, drive)
    if not os.path.isdir(drive_path): continue

    for seq in os.listdir(drive_path):
        seq_path = os.path.join(drive_path, seq)
        video_path = os.path.join(seq_path, "video.hevc")
        if not os.path.exists(video_path): continue

        # Nome cartella di output
        folder_name = f"{drive}|{seq}"
        save_path = os.path.join(output_base, folder_name)
        os.makedirs(save_path, exist_ok=True)

        try:
            fr = FrameReader(video_path)
            total_frames = fr.frame_count
            video_fps = 20
            step = int(round(video_fps / fps_target))  # campionamento

            frame_idx = 0
            saved_idx = 1

            for i in range(0, total_frames, step):
                try:
                    img = fr.get(i, pix_fmt='rgb24')[0]  # (H, W, 3)
                    img = cv2.resize(img, output_size, interpolation=cv2.INTER_CUBIC)
                    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
                    filename = os.path.join(save_path, f"{saved_idx:05}.jpg")
                    cv2.imwrite(filename, img)
                    saved_idx += 1
                except Exception as e:
                    print(f"⚠️ Errore a frame {i} in {folder_name}: {e}")

        except Exception as e:
            print(f"❌ Errore con {video_path}: {e}")

In [ ]:
from google.colab import files
files.download("/content/b0c9d2329ad1606b_2018-07-29--11-17-20_4.zip")

In [ ]:
import shutil
import os

# Percorso della cartella da zippare
original_path = "/content/frames/b0c9d2329ad1606b|2018-07-29--11-17-20|4"

# Crea un nome sicuro per lo zip (niente '|')
safe_name = original_path.split("/")[-1].replace("|", "_")
zip_path = f"/content/{safe_name}.zip"

# Comprimi
shutil.make_archive(zip_path.replace(".zip", ""), 'zip', original_path)

print(f"✅ Creato: {zip_path}")